# Ejemplo de Proyecto Final: Clasificación de Tejidos con el Dataset Kather 2016
Este notebook muestra cómo utilizar aprendizaje profundo con MobileNetV2 para clasificar imágenes de tejidos histológicos del dataset **Kather_texture_2016_image_tiles_5000** en 8 clases.

## 1. Descripción del Proyecto
- Objetivo: clasificar imágenes de tejido en una de las 8 categorías del dataset Kather 2016.
- Categorías:
  1. Tumor
  2. Stroma
  3. Complex
  4. Lympho
  5. Debris
  6. Mucosa
  7. Adipose
  8. Empty
- Técnica: Transfer Learning con MobileNetV2.
- Dataset: Estructura con subcarpetas por clase.

In [ ]:
print("Clasificación de imágenes histológicas en 8 clases del dataset Kather 2016")

## 2. Preparación del conjunto de datos
- Usa carpetas con nombres `01_TUMOR`, `02_STROMA`, ..., `08_EMPTY`.
- Idealmente dividido en `/train` y `/validation` manualmente si no viene preparado.

In [ ]:
from pathlib import Path
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import matplotlib.pyplot as plt

# Parámetros
img_size = (160, 160)
batch_size = 32

# Rutas a carpetas
dataset_root = Path("kather_dataset")
train_dir = dataset_root / "train"
val_dir = dataset_root / "validation"

if not train_dir.exists() or not val_dir.exists():
    raise FileNotFoundError("No se ha encontrado kather_dataset/train y kather_dataset/validation. Prepara antes el dataset en carpetas por clase.")

# Generadores
train_gen = ImageDataGenerator(preprocessing_function=tf.keras.applications.mobilenet_v2.preprocess_input)
val_gen = ImageDataGenerator(preprocessing_function=tf.keras.applications.mobilenet_v2.preprocess_input)

train_data = train_gen.flow_from_directory(train_dir, target_size=img_size, batch_size=batch_size, class_mode='categorical')
val_data = val_gen.flow_from_directory(val_dir, target_size=img_size, batch_size=batch_size, class_mode='categorical')

## 3. Entrenamiento del modelo (MobileNetV2)

In [ ]:
base_model = tf.keras.applications.MobileNetV2(input_shape=img_size + (3,), include_top=False, weights='imagenet')
base_model.trainable = False

model = tf.keras.Sequential([
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(8, activation='softmax')  # 8 clases
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
history = model.fit(train_data, validation_data=val_data, epochs=10)

## 4. Evaluación y Visualización de Resultados

In [ ]:
# Precisión por época
plt.plot(history.history['accuracy'], label='Train')
plt.plot(history.history['val_accuracy'], label='Val')
plt.title('Precisión durante el entrenamiento')
plt.xlabel('Épocas')
plt.ylabel('Precisión')
plt.legend()
plt.show()

## 5. Conclusiones
- Transfer Learning acelera el entrenamiento y mejora el rendimiento con pocos datos.
- La clasificación multiclase de imágenes médicas es factible con herramientas estándar.
- Este modelo se puede refinar con más datos y fine-tuning.